In [14]:
import os
import xml.etree.ElementTree as ET
import uuid
import random

import pandas as pd
from cvat_sdk import make_client
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [15]:
CVAT_USER = os.getenv("CVAT_USER")
CVAT_PASSWORD = os.getenv("CVAT_PASSWORD")

In [16]:
rd = random.Random()
rd.seed(42)
gen_uuid = lambda: str(uuid.UUID(int=rd.getrandbits(128), version=4))

def is_img(pth):
    return os.path.splitext(pth)[-1].lower() in [".png", ".jpg", ".jpeg", ".bmp"]

In [17]:
def parse_text(txt):
    text = txt.lower()
    if txt == "true":
        return True
    elif txt == "false":
        return False
    else:
        return str(txt).strip("none")

def split_point(s):
    return s.split(";")[0].split(",")

def split_corners(s):
    return [float(j)  for i in s.split(";") for j in i.split(",")]

def process_img_node(node: ET.Element) -> list:
    img_attributes = {
        "id": gen_uuid(),
        "name": node.attrib["name"],
        "width": int(node.attrib["width"]),
        "height": int(node.attrib["height"])
    }

    boxes = []
    positions = []
    corners = []
    for child in node:
        attributes = {}
        for meta in child:
            attributes[meta.attrib["name"]] = parse_text(meta.text)

        obj = {
            "label": child.attrib["label"],
            **attributes
        }
        if child.tag == "points":
            if ";" in child.attrib["points"]:
                obj["corners"] = split_corners(child.attrib["points"])
                corners.append(obj)
            else:
                obj["x"], obj["y"] = [float(i) for i in split_point(child.attrib["points"])]
                positions.append(obj)
        elif "xtl" in child.attrib:
            obj["xtl"] = float(child.attrib["xtl"])
            obj["ytl"] = float(child.attrib["ytl"])
            obj["xbr"] = float(child.attrib["xbr"])
            obj["ybr"] = float(child.attrib["ybr"])
            boxes.append(obj)

    objects = {}
    for box in boxes:
        idx = f"{box['ID']}-{box['label']}"
        objects[idx] = box
    for pos in positions:
        idx = f"{pos['ID']}-{pos['label']}"
        if idx in objects:
            objects[idx]["x"] = pos["x"]
            objects[idx]["y"] = pos["y"]
        else:
            objects[idx] = pos
    
    for tmp_corners in corners:
        idx = f"{tmp_corners['ID']}-{tmp_corners['label']}"
        if idx in objects:
            objects[idx]["corners"] = tmp_corners["corners"]
        else:
            objects[idx] = tmp_corners
    
    for i in objects.values():
        i["img_id"] = img_attributes["id"]
        i["id"] = gen_uuid()

    return list(objects.values()), img_attributes

def process_meta_node(node: ET.Element) -> dict:
    meta = {}
    for child in node:
        if child.tag not in ["task", "job"]:
            continue
        for meta_field in child:
            if meta_field.tag == "segments":
                for segment_data in meta_field[0]:
                    if segment_data.tag != "url":
                        continue
                    url = segment_data.text
                    job_id = int(url.split("/")[-1])
                    with make_client('https://app.cvat.ai', credentials=(CVAT_USER, CVAT_PASSWORD)) as client:
                        job = client.jobs.retrieve(job_id)
                        task_id = job.task_id
                    job_url = f"https://app.cvat.ai/tasks/{task_id}/jobs/{job_id}"
                    meta["task_id"] = task_id
                    meta["job_id"] = job_id
                    meta["job_url"] = job_url

    return meta

def process_annotation_file(pth: str, batch: str = None, folder: str = None) -> list:
    tree = ET.parse(pth)
    root = tree.getroot()
    objects = []
    img_attributes = []
    meta = {}
    for child in root:
        if child.tag == "meta":
            meta = process_meta_node(child)
        elif child.tag == "image":
            tmp_objects, tmp_img_attributes = process_img_node(child)
            objects += tmp_objects
            img_attributes.append(tmp_img_attributes)

    for img_attrib in img_attributes:
        img_attrib["batch"] = batch

    for img_attrib in img_attributes:
        img_attrib["folder"] = folder
    
    for img_attrib in img_attributes:
        img_attrib.update(meta)

    return objects, img_attributes

In [18]:
# base_path = "../../../db/object-detection"
base_path = "../../../../data/ObjectDetectionV3_1"
batches = sorted([i for i in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, i))])
batches = list(filter(lambda x: not x.startswith("."), batches))

folders = {}
for batch in batches:
    tmp_pth = os.path.join(base_path, batch)
    folders[batch] = sorted([i for i in os.listdir(tmp_pth) if os.path.isdir(os.path.join(tmp_pth, i))])

In [19]:
imgs = {}
annotations = {}
total_img_count = 0
for batch in batches:
    for folder in folders[batch]:
        tmp_pth = os.path.join(base_path, batch, folder)
        batch_imgs = [i for i in os.listdir(tmp_pth) if is_img(i)]
        batch_annotation = [i for i in os.listdir(tmp_pth) if i.endswith(".xml")][0]
        
        key = (batch, folder)
        imgs[key] = batch_imgs
        annotations[key] = batch_annotation
        total_img_count += len(batch_imgs)

print("total images:", total_img_count)

total images: 710


In [20]:
objects = []
img_attributes = []

for batch, annot in annotations.items():
    tmp_objects, tmp_img_attributes = process_annotation_file(os.path.join(base_path, *batch, annot), *batch)
    objects += tmp_objects
    img_attributes += tmp_img_attributes

df = pd.DataFrame(objects)
df_imgs = pd.DataFrame(img_attributes)
print(df_imgs.shape)

for (batch, folder), batch_imgs in imgs.items():
    for img in batch_imgs:
        if ((df_imgs["name"] == img) & (df_imgs["batch"] == batch) & (df_imgs["folder"] == folder)).any():
            continue

        df_imgs = pd.concat([df_imgs, pd.DataFrame([{"id": gen_uuid(), "name": img, "batch": batch, "folder": folder}])], ignore_index=True)

(710, 9)


In [21]:
df_imgs[df_imgs["job_url"].isna()]["folder"].unique()

array([], dtype=object)

In [22]:
df_imgs.head()

,id,name,width,height,batch,folder,task_id,job_id,job_url
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam2_H255_discaunt_29012024_2024-02-06-11h11m0...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182
1,b74d0fb1-32e7-4629-8fad-c1a606cb0fb3,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182
2,1a2a73ed-562b-4f79-8374-59eef50bea63,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182
3,e2acf72f-9e57-4f7a-a0ee-89aed453dd32,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182
4,29a3b2e9-5d65-4441-9588-42dea2bc372f,Cam2_H255_discaunt_29012024_2024-02-06-11h11m2...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182


In [23]:
print("total number of images: ", df_imgs.shape[0])

print("total number of uniques images with annotations: ", df["img_id"].unique().shape[0])

print("total number of annotated objects: ", df.shape[0])

object_per_img = df.groupby("img_id").count()["id"].values
print(f"objects per img: min = {object_per_img.min()}. max = {object_per_img.max()}. avg = {object_per_img.mean():.02f}")

print("number of object per class:\n", df["label"].value_counts())

total number of images:  710
total number of uniques images with annotations:  706
total number of annotated objects:  21650
objects per img: min = 1. max = 95. avg = 30.67
number of object per class:
 label
office_chair               6506
desk                       2869
desk_FULL                  2355
monitor                    1968
laptop_opened              1648
keyboard                   1067
plants                     1026
cabinet_FULL                706
bag                         630
cabinet                     589
bar_stool                   496
coffee_table                320
coat                        291
coffee_table_FULL           255
lounge_chair                197
whiteboard                  184
meeting_room_table          136
sofa                        135
laptop_closed               131
dinning_chair                77
sink                         32
signage                      14
printer                      11
meeting_room_table_FULL       4
coffee_station          

In [24]:
tmp_df = df[(df["xtl"].isna()) | (df["x"].isna())]
tmp_df = pd.merge(tmp_df, df_imgs, how="left", left_on="img_id", right_on="id")[["job_url", "batch", "folder", "name", "label", "ID", "xtl", "ytl", "xbr", "ybr", "x", "y"]]
tmp_df.to_csv(os.path.join(base_path, "no_boxes_or_position.csv"), index=False)
print("images with no boxes or position:", tmp_df["name"].unique().shape[0])

tmp_df = pd.merge(df, df_imgs, how="outer", left_on="img_id", right_on="id")
tmp_df = tmp_df[tmp_df["id_x"].isna()][["job_url", "batch", "folder","name"]].sort_values(by=["batch", "folder"])
tmp_df.to_csv(os.path.join(base_path, "no_annotations.csv"), index=False)
print("images with no annotations:", tmp_df["name"].unique().shape[0])

df1 = df_imgs.groupby(['name','batch', "folder"]).size().reset_index()[['name','batch', "folder"]]
duplicates = df1[df1.duplicated('name', keep=False)]
duplicate_info = duplicates.groupby('name').agg({
    'folder': lambda x: ', '.join(x),
    'batch': lambda x: ', '.join(x)
}).reset_index()
duplicate_info.to_csv(os.path.join(base_path, "duplicates.csv"), index=False)
print("duplicates:", duplicate_info.shape[0])

images with no boxes or position: 40
images with no annotations: 4
duplicates: 0


In [25]:
df.to_csv(os.path.join(base_path, "annotation.csv"), index=False)
df_imgs.to_csv(os.path.join(base_path, "images.csv"), index=False)